#### BoW: Ejemplos con reviews en español

**Bag of Words (BoW)** es una técnica fundamental de procesamiento de lenguaje natural (NLP) para convertir textos en datos numéricos.

**¿Cómo funciona?**
- Cada palabra se convierte en una variable/característica
- Los valores pueden ser:
  - **Frecuencia**: cuántas veces aparece la palabra (0, 1, 2, ...)
  - **Binarios**: presencia/ausencia (0 o 1)
  - **TF-IDF**: peso que considera la importancia de la palabra en el corpus

**Ventajas:**
- Simple y rápido
- Funciona bien para clasificación de textos
- Base para técnicas más complejas

**Limitaciones:**
- Pierde orden de palabras (contexto)
- No captura semántica profunda

##### Ejemplo 1: Bag of Words Simple con CountVectorizer

En este ejemplo vamos a:
1. Crear 3 documentos de texto simples
2. Usar `CountVectorizer` para convertirlos en una matriz numérica
3. Cada columna representa una palabra única
4. Cada valor es la **frecuencia** (cuántas veces aparece esa palabra en ese documento)
5. Combinar con metadatos adicionales

**Resultado esperado:** Una matriz donde filas=documentos y columnas=palabras del vocabulario

In [ ]:
# ============================================
# IMPORTAR LIBRERÍAS NECESARIAS
# ============================================

import pandas as pd  # Manipulación de dataframes
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer  # Vectorización de texto
from nltk.corpus import stopwords  # Palabras vacías (stopwords)
import nltk  # Natural Language Toolkit
import numpy as np  # Operaciones numéricas

# Descargar recursos de NLTK
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\tomas\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
# Paso 1: Crear lista de documentos de ejemplo
documents = [
    "El gato está en el tejado",      
    "El perro está en el jardín",     
    "El gato y el perro son amigos"   
]

# Paso 2: Crear un vectorizador Bag of Words
# CountVectorizer cuenta la frecuencia de cada palabra en cada documento
vectorizer = CountVectorizer()

# Paso 3: Ajustar a los documentos y transformarlos a matriz numérica
# fit_transform() aprende el vocabulario y convierte documentos a matriz
bow_matrix = vectorizer.fit_transform(documents)

# Paso 4: Obtener los nombres de las características (palabras del vocabulario)
feature_names = vectorizer.get_feature_names_out()

# Paso 5: Convertir la matriz sparse a DataFrame para visualización
bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=feature_names
)

print("Matriz Bag of Words (frecuencia de palabras por documento):")
print(bow_df)

# Paso 6: Crear DataFrame adicional con metadatos
additional_data = {
    'author': ['Juan', 'Ana', 'Pedro'],
    'length': [5, 5, 6]
}
additional_df = pd.DataFrame(additional_data)

# Paso 7: Combinar metadatos con matriz BoW
combined_df = pd.concat([additional_df, bow_df], axis=1)

print("\nDataFrame combinado (metadatos + matriz BoW):")
print(combined_df)

Matriz Bag of Words (frecuencia de palabras por documento):
   amigos  el  en  está  gato  jardín  perro  son  tejado
0       0   2   1     1     1       0      0    0       1
1       0   2   1     1     0       1      1    0       0
2       1   2   0     0     1       0      1    1       0

DataFrame combinado (metadatos + matriz BoW):
  author  length  amigos  el  en  está  gato  jardín  perro  son  tejado
0   Juan       5       0   2   1     1     1       0      0    0       1
1    Ana       5       0   2   1     1     0       1      1    0       0
2  Pedro       6       1   2   0     0     1       0      1    1       0


##### Ejemplo 2: TF-IDF con Dataset Real (Reseñas de Películas)

En este ejemplo usaremos un **dataset real** de reseñas de películas en español.

**Qué haremos:**
1. Cargar datos desde CSV
2. Aplicar **TF-IDF** en lugar de frecuencia simple
3. Explorar el dataset filtrando por película
4. Vectorizar reseñas y crear matriz numérica
5. Combinar datos originales + características TF-IDF
6. Guardar resultado en archivo

**¿Por qué TF-IDF y no Count?**
- **TF-IDF** da más peso a palabras raras pero significativas
- **Count** solo cuenta frecuencia (palabras comunes dominan)
- TF-IDF es mejor para análisis de contenido

**Dataset:** Reseñas de FilmAffinity  
Fuente: https://www.kaggle.com/datasets/pshiju/djinn-spanish-text-vector

In [ ]:
# ============================================
# PASO 1: CARGAR DATASET DE RESEÑAS
# ============================================

# Ruta al archivo CSV con reseñas
path = 'C:\\Users\\tomas\\ML\\env1\\data\\reviews_filmaffinity.csv'

# Cargar datos: separador "||", primera fila es header
df2 = pd.read_table(path, sep='\|\|', header=0, engine='python')

# Mostrar 5 muestras aleatorias
print("Primeras reseñas del dataset:")
print(df2.sample(5))

Primeras reseñas del dataset:
                                    film_name     gender film_avg_rate  \
3465                                Alatriste  Aventuras           5,5   
4305                                Celda 211   Thriller           7,7   
3872                                Celda 211   Thriller           7,7   
1538  La gran aventura de Mortadelo y Filemón    Comedia           4,8   
4711                    Lo dejo cuando quiera    Comedia           5,2   

      review_rate                                   review_title  \
3465          8.0                       más arte que espectáculo   
4305          1.0  Un bodrio de calidad para suerte de las masas   
3872          6.0                            ¿Y si fuera yankee?   
1538          8.0                                    Que pena...   
4711          7.0                              Dando en el clavo   

                                            review_text  
3465  Una pena no haber leido el libro, pero al ver ...  


<>:9: SyntaxWarning: invalid escape sequence '\|'
<>:9: SyntaxWarning: invalid escape sequence '\|'
C:\Users\tomas\AppData\Local\Temp\ipykernel_34600\3683309576.py:9: SyntaxWarning: invalid escape sequence '\|'
  df2 = pd.read_table(path, sep='\|\|', header=0, engine='python')


In [ ]:
# ============================================
# PASO 2: PREPARAR VECTORIZADOR CON STOPWORDS
# ============================================

# Obtener palabras vacías (stopwords) en español
# Stopwords: palabras comunes que no aportan significado ("el", "la", "de", "y", etc)
spanish_stopwords = stopwords.words('spanish')

# Crear vectorizador TF-IDF que excluya stopwords
# TF-IDF = Term Frequency × Inverse Document Frequency
# Da más peso a palabras raras pero significativas
vectorizer = TfidfVectorizer(stop_words=spanish_stopwords)

print(f"Total de stopwords en español: {len(spanish_stopwords)}")
print(f"Primeros 20 stopwords: {spanish_stopwords[:20]}")

Total de stopwords en español: 313
Primeros 20 stopwords: ['de', 'la', 'que', 'el', 'en', 'y', 'a', 'los', 'del', 'se', 'las', 'por', 'un', 'para', 'con', 'no', 'una', 'su', 'al', 'lo']


In [ ]:
# Ver lista completa de stopwords
print(f"\nTodos los {len(spanish_stopwords)} stopwords en español:")
print(spanish_stopwords)


Todos los 313 stopwords en español:
['de', 'la', 'que', 'el', 'en', 'y', 'a', 'los', 'del', 'se', 'las', 'por', 'un', 'para', 'con', 'no', 'una', 'su', 'al', 'lo', 'como', 'más', 'pero', 'sus', 'le', 'ya', 'o', 'este', 'sí', 'porque', 'esta', 'entre', 'cuando', 'muy', 'sin', 'sobre', 'también', 'me', 'hasta', 'hay', 'donde', 'quien', 'desde', 'todo', 'nos', 'durante', 'todos', 'uno', 'les', 'ni', 'contra', 'otros', 'ese', 'eso', 'ante', 'ellos', 'e', 'esto', 'mí', 'antes', 'algunos', 'qué', 'unos', 'yo', 'otro', 'otras', 'otra', 'él', 'tanto', 'esa', 'estos', 'mucho', 'quienes', 'nada', 'muchos', 'cual', 'poco', 'ella', 'estar', 'estas', 'algunas', 'algo', 'nosotros', 'mi', 'mis', 'tú', 'te', 'ti', 'tu', 'tus', 'ellas', 'nosotras', 'vosotros', 'vosotras', 'os', 'mío', 'mía', 'míos', 'mías', 'tuyo', 'tuya', 'tuyos', 'tuyas', 'suyo', 'suya', 'suyos', 'suyas', 'nuestro', 'nuestra', 'nuestros', 'nuestras', 'vuestro', 'vuestra', 'vuestros', 'vuestras', 'esos', 'esas', 'estoy', 'estás', 'es

In [ ]:
# ============================================
# PASO 3: EXPLORAR DATASET - RESEÑAS POR PELÍCULA
# ============================================

# Contar cuántas reseñas tiene cada película
films = df2.groupby(['film_name']).count()

print("Número de reseñas por película:")
print(films)

Número de reseñas por película:
                                         gender  film_avg_rate  review_rate  \
film_name                                                                     
Ahora o nunca                                81             81           81   
AirBag                                       98             98           98   
Alatriste                                   445            445          445   
Atrapa la bandera                            51             51           51   
Campeones                                   199            199          199   
Celda 211                                   487            487          487   
Días de fútbol                               57             57           57   
El bola                                      64             64           64   
El laberinto del fauno                      495            495          495   
El mejor verano de mi vida                   54             54           54   
El orfanato         

In [ ]:
# ============================================
# PASO 4: FILTRAR DATOS DE UNA PELÍCULA ESPECÍFICA
# ============================================

# Seleccionar todas las reseñas de "Tadeo Jones 2"
df3 = df2[df2['film_name'] == 'Tadeo Jones 2']

print(f"Total de reseñas de 'Tadeo Jones 2': {len(df3)}")
print("\nPrimeras reseñas:")
print(df3)

Total de reseñas de 'Tadeo Jones 2': 36

Primeras reseñas:
          film_name     gender film_avg_rate  review_rate  \
3236  Tadeo Jones 2  Animación           5,7          7.0   
3237  Tadeo Jones 2  Animación           5,7          6.0   
3238  Tadeo Jones 2  Animación           5,7          8.0   
3239  Tadeo Jones 2  Animación           5,7          6.0   
3240  Tadeo Jones 2  Animación           5,7          4.0   
3241  Tadeo Jones 2  Animación           5,7          8.0   
3242  Tadeo Jones 2  Animación           5,7          8.0   
3243  Tadeo Jones 2  Animación           5,7          6.0   
3244  Tadeo Jones 2  Animación           5,7          7.0   
3245  Tadeo Jones 2  Animación           5,7          5.0   
3246  Tadeo Jones 2  Animación           5,7          6.0   
3247  Tadeo Jones 2  Animación           5,7          6.0   
3248  Tadeo Jones 2  Animación           5,7          6.0   
3249  Tadeo Jones 2  Animación           5,7          9.0   
3250  Tadeo Jones 2  Anima

In [ ]:
# ============================================
# PASO 5: VECTORIZAR RESEÑAS CON TF-IDF
# ============================================

# Paso 5a: Extraer textos de reseñas como lista
reviews = df3['review_text'].tolist()

print(f"Total de reseñas a vectorizar: {len(reviews)}")

# Paso 5b: Aplicar vectorizador TF-IDF a todas las reseñas
# fit_transform() aprende vocabulario y transforma a matriz
tfidf_matrix3 = vectorizer.fit_transform(reviews)

print(f"Dimensiones matriz TF-IDF: {tfidf_matrix3.shape}")
print(f"  - Filas: {tfidf_matrix3.shape[0]} (reseñas)")
print(f"  - Columnas: {tfidf_matrix3.shape[1]} (palabras únicas)")

# Paso 5c: Convertir matriz sparse a DataFrame para visualización
# (Las matrices sparse ahorran memoria no guardando ceros)
tfidf_df3 = pd.DataFrame(
    tfidf_matrix3.toarray(),
    columns=vectorizer.get_feature_names_out()
)

print("\nPrimeras filas de la matriz TF-IDF:")

Total de reseñas a vectorizar: 36
Dimensiones matriz TF-IDF: (36, 1874)
  - Filas: 36 (reseñas)
  - Columnas: 1874 (palabras únicas)

Primeras filas de la matriz TF-IDF:


In [ ]:
# Mostrar las primeras 20 reseñas con sus vectores TF-IDF
print("Matriz TF-IDF de las primeras 20 reseñas:")
print(tfidf_df3.head(20))

Matriz TF-IDF de las primeras 20 reseñas:
         000   05        10      100  10retalesdeacetato   10verde  150  \
0   0.000000  0.0  0.000000  0.00000            0.000000  0.000000  0.0   
1   0.000000  0.0  0.000000  0.00000            0.000000  0.000000  0.0   
2   0.000000  0.0  0.000000  0.00000            0.000000  0.000000  0.0   
3   0.000000  0.0  0.000000  0.00000            0.000000  0.000000  0.0   
4   0.000000  0.0  0.000000  0.00000            0.000000  0.000000  0.0   
5   0.000000  0.0  0.053793  0.00000            0.000000  0.000000  0.0   
6   0.000000  0.0  0.056411  0.00000            0.000000  0.000000  0.0   
7   0.000000  0.0  0.000000  0.00000            0.000000  0.000000  0.0   
8   0.000000  0.0  0.000000  0.00000            0.000000  0.000000  0.0   
9   0.000000  0.0  0.000000  0.00000            0.000000  0.000000  0.0   
10  0.000000  0.0  0.000000  0.00000            0.000000  0.000000  0.0   
11  0.000000  0.0  0.000000  0.00000            0.000000  

In [ ]:
# Ver todas las palabras (features) que se extrajeron
print(f"Total de palabras únicas en el corpus: {len(tfidf_df3.columns)}")
print("\nPalabras extraídas:")
print(tfidf_df3.columns.tolist())

Total de palabras únicas en el corpus: 1874

Palabras extraídas:
['000', '05', '10', '100', '10retalesdeacetato', '10verde', '150', '18', '2009', '2011', '2012', '2015', '2016', '2018', '27', '2d', '3d', '51', '600', '80', '81', '90', 'abandona', 'abierto', 'absoluta', 'absolutamente', 'absurdos', 'aburrida', 'aburrí', 'acaba', 'acabar', 'acabo', 'acabó', 'acción', 'acertada', 'acertados', 'acierto', 'acompaña', 'acompañado', 'acompañan', 'acompañaras', 'acorde', 'actor', 'acudir', 'adecuada', 'adelantaba', 'adelante', 'además', 'adinerada', 'adrian', 'adultos', 'afeitarse', 'afirma', 'agarre', 'agotamiento', 'agradable', 'agradece', 'agradecen', 'agradecer', 'aguas', 'ahora', 'ahí', 'ajena', 'albañil', 'alberto', 'alguna', 'alguno', 'algún', 'alhambra', 'aliciente', 'alimón', 'allá', 'alma', 'alonso', 'alrededor', 'alta', 'altamente', 'alto', 'altura', 'amante', 'ambas', 'ambienta', 'ambientación', 'ambientes', 'ambulante', 'amena', 'amenaza', 'americana', 'americanas', 'amiga', 'amig

In [ ]:
# ============================================
# PASO 6: COMBINAR DATOS ORIGINALES + VECTORES TF-IDF
# ============================================

# Paso 6a: Crear columna clave para hacer merge
# (Los índices podrían no coincidir después de transformaciones)
df3['key'] = range(len(df3))
tfidf_df3['key'] = range(len(tfidf_df3))

# Paso 6b: Hacer MERGE (unión) de datos
# Combinamos: reseña original + rating + todas las características TF-IDF
combined_df3 = pd.merge(
    df3,          # Datos originales (texto, rating, fecha, etc)
    tfidf_df3,    # Vectores TF-IDF
    on='key',     # Unir por columna clave
    how='left'    # Mantener todos registros de df3
)

print("DataFrame final con datos originales + vectores TF-IDF:")
print(f"Dimensiones: {combined_df3.shape} (reseñas x características)")
print("\nPrimeras 3 columnas de primeras 3 filas:")
print(combined_df3.iloc[:3, :3])

DataFrame final con datos originales + vectores TF-IDF:
Dimensiones: (36, 1881) (reseñas x características)

Primeras 3 columnas de primeras 3 filas:
       film_name     gender film_avg_rate
0  Tadeo Jones 2  Animación           5,7
1  Tadeo Jones 2  Animación           5,7
2  Tadeo Jones 2  Animación           5,7


C:\Users\tomas\AppData\Local\Temp\ipykernel_34600\3016885273.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df3['key'] = range(len(df3))


In [ ]:
# Mostrar una reseña específica completa
# Útil para ver cómo se ve una fila entera con todas sus características
print("Reseña número 32 (ejemplo completo - primeras 10 columnas):")
print(combined_df3.iloc[32, :10])

Reseña número 32 (ejemplo completo - primeras 10 columnas):
film_name                                            Tadeo Jones 2
gender                                                   Animación
film_avg_rate                                                  5,7
review_rate                                                    1.0
review_title                                                 SOPOR
review_text      Ayer estuve viendo con mis 2 hijos la nueva pe...
key                                                             32
000                                                            0.0
05                                                             0.0
10                                                             0.0
Name: 32, dtype: object


In [ ]:
# ============================================
# PASO 7: GUARDAR RESULTADO EN ARCHIVO
# ============================================

# Guardar DataFrame completo (datos + vectores TF-IDF) en archivo Excel
# Este archivo contiene: texto original + todas las características numéricas
combined_df3.to_csv('tadeo.xlsx', index=False)

print("✓ Archivo 'tadeo.xlsx' guardado exitosamente")
print(f"  - Reseñas: {combined_df3.shape[0]}")
print(f"  - Características totales: {combined_df3.shape[1]}")

In [ ]:
# ============================================
# PASO FINAL: VISUALIZAR MATRIZ DENSA
# ============================================

# Convertir matriz sparse a matriz densa para visualización completa
# Nota: 
#   - Matrices SPARSE: ahorran memoria (no guardan ceros)
#   - Matrices DENSAS: muestran todos los valores (incluyendo ceros)
dense_tfidf_matrix = tfidf_matrix3.todense()

print("Matriz TF-IDF DENSA (con todos los ceros explícitos):")
print(f"Dimensiones: {dense_tfidf_matrix.shape} (reseñas x palabras)")
print(f"\nPrimeras 5 filas × primeras 10 columnas:")
print(dense_tfidf_matrix[:5, :10])

Matriz TF-IDF:
[[0 0 0 ... 2 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 1 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


---

## Resumen de lo que hemos aprendido

| Paso | Acción | Resultado |
|------|--------|-----------|
| 1️⃣ | Cargamos dataset de reseñas | 📊 DataFrame con textos originales |
| 2️⃣ | Creamos vectorizador TF-IDF | 🔧 Herramienta para convertir textos a números |
| 3️⃣ | Aplicamos TF-IDF a reseñas | 📈 Matriz de características (sparse) |
| 4️⃣ | Convertimos a DataFrame | 📋 Visualización clara de vectores |
| 5️⃣ | Hicimos MERGE de datos + vectores | 🔗 Tabla completa: original + numérico |
| 6️⃣ | Guardamos en archivo | 💾 tadeo.xlsx listo para ML |

**¿Por qué es útil esto?**
- Los modelos de ML necesitan números, no textos
- TF-IDF captura palabras importantes automáticamente
- Ahora podemos hacer: clasificación, clustering, búsqueda, etc

**Próximos pasos:**
- Usar estas características para entrenar un clasificador
- Hacer análisis de similitud entre reseñas
- Detectar temas comunes en reseñas